# **Прототипные сети: «это похоже вот на то»**### Практика к уроку про ProtoPNet, блок «Нелинейные интерпретируемые модели»Здесь мы соберем ProtoPNet целиком и своими руками — на задаче, которая считаетсяна процессоре за пару минут: три класса Fashion-MNIST и по два прототипа на класс.Разница с остальными методами курса в том, что объяснение здесь не строится после ответаи не выводится из структуры. Оно **и есть само вычисление**: сеть складывает прогнозиз сходств с образцами, и эти сходства можно показать читательнице.По шагам мы: обучим основу вместе с прототипами, проведем **push** — операцию, ради которойвсе и затевалось, — и посмотрим, что она делает с точностью. Затем настроим один последнийслой и прочитаем объяснение конкретного снимка как «сходство x вес».Приятной работы!

In [ ]:
import numpy as npimport torchimport torch.nn as nnimport torch.nn.functional as Fimport matplotlib.pyplot as pltfrom torchvision import datasets, transformstorch.manual_seed(0)np.random.seed(0)CLASSES = [0, 1, 9]          # t-shirt, trousers, ankle bootPROTO_PER_CLASS = 2          # прототипов на классD = 64                       # длина вектора клетки и прототипаEPS = 1e-4

## ДанныеБерем три непохожих класса, чтобы прототипы были различимы глазом: футболка, брюкии ботинок. По 600 снимков на класс в обучении и по 150 в тесте — этого хватает,чтобы увидеть все эффекты, и мало, чтобы ждать.

In [ ]:
tf = transforms.Compose([transforms.Resize(64), transforms.ToTensor()])train_full = datasets.FashionMNIST('./data', train=True, download=True, transform=tf)test_full = datasets.FashionMNIST('./data', train=False, download=True, transform=tf)def subset(ds, n_per_class):    idx, seen = [], {c: 0 for c in CLASSES}    for i, y in enumerate(ds.targets.tolist()):        if y in seen and seen[y] < n_per_class:            idx.append(i); seen[y] += 1        if all(v >= n_per_class for v in seen.values()):            break    return torch.utils.data.Subset(ds, idx)train_ds, test_ds = subset(train_full, 600), subset(test_full, 150)remap = {c: i for i, c in enumerate(CLASSES)}loader = lambda ds, bs, sh: torch.utils.data.DataLoader(ds, batch_size=bs, shuffle=sh)print(len(train_ds), len(test_ds))

## МодельТри части, ровно как в уроке.**Основа $f$** — сверточная сеть без классифицирующей головы. На выходе тензор $D \times H\times W$: каждая из $H \cdot W$ клеток описывает свой участок снимка вектором длины $D$.**Прототипный слой** — $m$ обучаемых векторов той же длины $D$, по два на класс. Каждыйзакреплен за своим классом.**Линейный слой $h$** берет $m$ сходств и выдает оценки классов. Веса стартуют не сослучайных: $1$ для прототипов своего класса и $-0.5$ для чужих — это сразу задает нужноечтение объяснения.Сходство считается по формуле из урока:$$g_{p_j}(z) = \max_{\tilde z} \log\left(\frac{\|\tilde z - p_j\|^2 + 1}{\|\tilde z - p_j\|^2 + \varepsilon}\right)$$Логарифм убывает по расстоянию, а максимум по клеткам отвечает на вопрос «нашелся лигде-нибудь на снимке участок, похожий на этот образец» — и запоминает, где именно.

In [ ]:
class ProtoPNet(nn.Module):    def __init__(self, n_classes, per_class):        super().__init__()        self.n_classes, self.per_class = n_classes, per_class        self.m = n_classes * per_class        # свёрточная основа f: на выходе карта D x H x W, каждая клетка — «кусочек картинки»        self.features = nn.Sequential(            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),            nn.Conv2d(64, D, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),            nn.Conv2d(D, D, 1), nn.Sigmoid(),        )        # прототипный слой: m обучаемых векторов, по per_class на каждый класс        self.prototypes = nn.Parameter(torch.rand(self.m, D))        self.identity = torch.zeros(self.m, n_classes)        for j in range(self.m):            self.identity[j, j // per_class] = 1        # линейный слой h: 1 для своих прототипов, -0.5 для чужих        self.last = nn.Linear(self.m, n_classes, bias=False)        with torch.no_grad():            self.last.weight.copy_((self.identity * 1.5 - 0.5).t())    def similarity(self, z):        B, _, H, W = z.shape        patches = z.permute(0, 2, 3, 1).reshape(B, H * W, D)        d2 = torch.cdist(patches, self.prototypes.unsqueeze(0).expand(B, -1, -1)) ** 2        sim = # Ваш код здесь — формула сходства из урока: log((d2 + 1) / (d2 + EPS))        return sim, d2    def forward(self, x):        z = self.features(x)        sim, d2 = self.similarity(z)        pooled = sim.max(dim=1).values      # максимум по клеткам: нашёлся ли похожий участок        return self.last(pooled), pooled, d2, zmodel = ProtoPNet(len(CLASSES), PROTO_PER_CLASS)sum(p.numel() for p in model.parameters())

## Стадия 1: основа и прототипыК кросс-энтропии добавлены два слагаемых из работы Chen et al.:- **кластеризующее** — у каждого снимка должна найтись клетка близко хотя бы к одному  прототипу своего класса;- **разделяющее** — от чужих прототипов клетки надо держать подальше.Первое собирает прототипы в осмысленные образцы, второе не дает им стать общими для всехклассов.

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3)for epoch in range(3):    model.train(); tot = correct = 0    for x, y in loader(train_ds, 64, True):        y = torch.tensor([remap[int(v)] for v in y])        logits, pooled, d2, _ = model(x)        ce = F.cross_entropy(logits, y)        ident = model.identity[:, y].t()          # 1 у прототипов своего класса        min_d = d2.min(dim=1).values              # ближайшая клетка к каждому прототипу        clst = (min_d * ident).sum(1).mean()      # кластеризующее: свои — ближе        sep = # Ваш код здесь — разделяющее слагаемое: чужие прототипы держим дальше        loss = ce + 0.8 * clst - 0.08 * sep        opt.zero_grad(); loss.backward(); opt.step()        tot += len(y); correct += (logits.argmax(1) == y).sum().item()    print(f'эпоха {epoch + 1}: accuracy {correct / tot:.3f}')

In [ ]:
@torch.no_grad()def accuracy(ds):    model.eval(); ok = n = 0    for x, y in loader(ds, 128, False):        y = torch.tensor([remap[int(v)] for v in y])        ok += (model(x)[0].argmax(1) == y).sum().item(); n += len(y)    return ok / nacc_before = accuracy(test_ds)print(f'до push: {acc_before:.3f}')

## Push: прототип становится куском реального снимкаПока прототип — просто обучаемый вектор, показать читательнице нечего: «сходство 0.91с вектором $p_1$» не объяснение. Push заменяет каждый прототип **ближайшей к нему клеткойсреди обучающих снимков его класса**.После этого прототип перестает быть абстракцией: это конкретный участок конкретного снимка,который можно вырезать и напечатать рядом с объяснением.Цена известна заранее — обратите внимание на точность до и после.

In [ ]:
@torch.no_grad()def push():    """Каждый прототип заменяется ближайшей клеткой обучающего снимка своего класса."""    model.eval()    best = {j: (float('inf'), None, None) for j in range(model.m)}    for x, y in loader(train_ds, 64, False):        y = torch.tensor([remap[int(v)] for v in y])        z = model.features(x)        B, _, H, W = z.shape        patches = z.permute(0, 2, 3, 1).reshape(B, H * W, D)        for j in range(model.m):            mask = (y == j // model.per_class)     # только снимки своего класса            if not mask.any():                continue            p = patches[mask]            d = ((p - model.prototypes[j]) ** 2).sum(-1)            v, flat = d.view(-1).min(0)            if v.item() < best[j][0]:                ni, cell = divmod(int(flat), H * W)                best[j] = (v.item(), p[ni, cell].clone(), (x[mask][ni], cell, H, W))    for j, (_, vec, _) in best.items():        model.prototypes.data[j] = vec            # прототип стал куском реального снимка    return bestsources = push()acc_after = accuracy(test_ds)print(f'после push: {acc_after:.3f}   (изменение {acc_after - acc_before:+.3f})')

## Стадия 3: только последний слойОснова и прототипы заморожены, настраивается один $h$ — задача выпуклая. $L_1$-штраф давитвеса **чужих** прототипов к нулю, и в объяснении остаются только доводы «за», безбухгалтерии из отрицательных вкладов.

In [ ]:
for p in model.features.parameters():    p.requires_grad = Falsemodel.prototypes.requires_grad = False            # основа и прототипы замороженыopt2 = torch.optim.Adam(model.last.parameters(), lr=1e-3)for epoch in range(2):    model.train()    for x, y in loader(train_ds, 64, True):        y = torch.tensor([remap[int(v)] for v in y])        logits, *_ = model(x)        l1 = # Ваш код здесь — L1-штраф на веса ЧУЖИХ прототипов: (1 - model.identity.t())        loss = F.cross_entropy(logits, y) + 1e-3 * l1        opt2.zero_grad(); loss.backward(); opt2.step()print(f'после дообучения: {accuracy(test_ds):.3f}')

## Читаем объяснениеВклад прототипа в прогноз — это **сходство, умноженное на вес** в последнем слое. Прототипысвоего класса дают плюс, чужого — минус: ровно так слой и был инициализирован.

In [ ]:
model.eval()x, y = test_ds[0]with torch.no_grad():    logits, pooled, _, _ = model(x.unsqueeze(0))pred = int(logits.argmax())contrib = (pooled[0] * model.last.weight[pred]).detach().numpy()order = np.argsort(-contrib)print(f'истинный класс: {CLASSES[remap[int(y)]]}, прогноз: {CLASSES[pred]}\n')for j in order[:4]:    own = 'свой' if j // PROTO_PER_CLASS == pred else 'чужой'    print(f'прототип {j} ({own} класс): сходство {pooled[0][j]:6.3f} '          f'x вес {model.last.weight[pred][j]:6.3f} = {contrib[j]:6.3f}')

## Показываем, на что похожеТри картинки рядом: сам снимок, карта сходства с лучшим прототипом (где именно сеть нашлапохожий участок) и тот участок обучающего снимка, из которого этот прототип получился при push.Это и есть объяснение вида «вот эта часть вашего снимка похожа вот на этот образец».

In [ ]:
with torch.no_grad():    z = model.features(x.unsqueeze(0))    sim, _ = model.similarity(z)H = W = z.shape[-1]j = int(order[0])smap = sim[0, :, j].reshape(H, W).numpy()src_img, src_cell, sH, sW = sources[j][2]r, c = divmod(src_cell, sW)step = src_img.shape[-1] // sHfig, ax = plt.subplots(1, 3, figsize=(11, 3.6))ax[0].imshow(x.squeeze(), cmap='gray'); ax[0].set_title('объясняемый снимок')ax[1].imshow(x.squeeze(), cmap='gray')ax[1].imshow(np.kron(smap, np.ones((step, step))), alpha=0.5, cmap='jet')ax[1].set_title(f'сходство с прототипом {j}')ax[2].imshow(src_img.squeeze()[r * step:(r + 1) * step, c * step:(c + 1) * step], cmap='gray')ax[2].set_title('участок-источник прототипа')for a in ax:    a.axis('off')plt.tight_layout(); plt.show()

## Задания1. **Просадка после push.** Сравните точность до и после push и объясните знак разницы.   Почему прототип, ставший куском реального снимка, ухудшает прогноз — и почему это   не считается поломкой?2. **Цена интерпретируемости.** Обучите ту же основу с обычной линейной головой вместо   прототипного слоя (замените `ProtoPNet` на `nn.Sequential(features, Flatten, Linear)`).   На сколько отличается точность? Это и есть плата за объяснимость по построению.3. **Число прототипов.** Поставьте `PROTO_PER_CLASS = 1` и `= 4`. Как меняются точность   и читаемость объяснения? Где, по-вашему, компромисс?4. **Чужие прототипы.** Посмотрите на веса `model.last.weight` после стадии 3. Сколько   весов чужих прототипов ушло к нулю? Что это дает объяснению?5. **Границы.** Возьмите снимок, на котором модель ошибается, и постройте для него то же   объяснение. На какие прототипы она опирается? Видно ли из объяснения, почему она ошиблась?6. **Проверка на возмущении.** Сдвиньте снимок на два-три пикселя (`torch.roll`) и пересчитайте   сходства. Насколько они изменились? Это ровно та проверка, которую предлагают   Hoffmann et al. (2021) в работе «This Looks Like That… Does it?».

## Что стоит унестиПрототипное объяснение честнее многих: оно не достраивается снаружи, а совпадаетс вычислением. Но у него есть своя граница, и она там же, где сила.**Сходство считается в латентном пространстве сети, и совпадать с человеческим ононе обязано.** Рамка на карте сходства показывает, где максимум, но не говорит, чем именнопрототип зацепился — цветом, формой или текстурой. Это подсказка, а не измерение.Проверять такие объяснения предлагают возмущениями — задание 6 как раз про это.